# Session 8 Machine Learning Evaluation

**Learning goals:**

- select and interpret suitable evaluation metrics for classification, regression and clustering
- explain the limitations of individual metrics and relate metric choice to the purpose of a model
- identify underfitting and overfitting by comparing training and validation performance
- use k-fold cross-validation to estimate how consistently a model generalises
- keep preprocessing and model evaluation separate from the final test set

**Prerequisites:**

- familiarity with classification, regression and clustering
- basic Python, pandas, NumPy, Matplotlib and scikit-learn
- experience fitting a scikit-learn model and using a train/test split

Model evaluation asks more than whether a score is large or small. A useful evaluation connects the model's task, the consequences of its errors and evidence from data the model did not use for training.


## Outline

1. Evaluation foundations
2. Classification evaluation metrics
3. Regression evaluation metrics
4. Clustering evaluation metrics
5. Underfitting and overfitting
6. K-fold cross-validation


## 1. Evaluation Foundations

The correct evaluation method depends on the question the model answers:

| Task | Model output | Main evaluation question |
|---|---|---|
| Classification | a category or class probability | Which cases are classified correctly, and which errors matter? |
| Regression | a continuous number | How large are the prediction errors? |
| Clustering | groups discovered without target labels | Are the groups compact, separated, stable and useful? |

For supervised learning, data normally has three distinct roles:

- **Training data** fits model parameters.
- **Validation data** supports model selection and tuning.
- **Test data** provides one final estimate after model choices have been made.

A **baseline** is a simple reference, such as always predicting the most frequent class or the mean target. A more complex model should provide meaningful improvement over that reference.

Evaluation is unreliable when information from validation or test observations influences training. This is called **data leakage**. Preprocessing that learns from data, such as scaling or imputation, must therefore be fitted using training data only. A scikit-learn pipeline helps enforce this separation.


## 2. Classification Evaluation Metrics

Classification metrics are calculated from predicted classes and known labels. Before calculating them, compare the practical meaning of each metric:

| Metric | Question it answers | Best value | When it is useful |
|---|---|---:|---|
| Accuracy | What share of all predictions are correct? | 1.0 | Classes are balanced and different mistakes have similar costs. |
| Precision | When the model predicts the positive class, how often is it correct? | 1.0 | False positives are costly. |
| Recall | Of the actual positive cases, how many did the model find? | 1.0 | False negatives are costly. |
| F1 score | How strong is the balance between precision and recall? | 1.0 | One summary score is needed and both false positives and false negatives matter. |

### How Classification Metrics Are Calculated

The calculation begins with four counts from the binary confusion matrix:

| Outcome | Meaning |
|---|---|
| True positive (TP) | a positive case predicted as positive |
| True negative (TN) | a negative case predicted as negative |
| False positive (FP) | a negative case incorrectly predicted as positive |
| False negative (FN) | a positive case incorrectly predicted as negative |

Substitute these counts into the following formulas:

$$
\text{Accuracy} = \frac{TP + TN}{TP + TN + FP + FN}
$$

$$
\text{Precision} = \frac{TP}{TP + FP}
$$

$$
\text{Recall} = \frac{TP}{TP + FN}
$$

$$
F_1 = 2 \times \frac{\text{Precision} \times \text{Recall}}{\text{Precision} + \text{Recall}}
$$

### Interpreting F1 Values

F1 ranges from **0 to 1**, and a value closer to 1 is better:

| F1 value | General meaning |
|---|---|
| 1.00 | perfect precision and perfect recall |
| 0.80–0.99 | often described as strong, provided both precision and recall meet the task requirements |
| 0.60–0.79 | may be useful, but important errors and the baseline require careful review |
| Below 0.60 | often indicates substantial classification errors |
| 0.00 | precision, recall or both provide no successful positive-class performance |

These bands are an **informal classroom guide, not a universal acceptance standard**. An F1 score above 0.80 is commonly treated as a promising result, but it is acceptable only when it improves meaningfully on the baseline and the underlying precision and recall satisfy the real error costs. In a high-risk screening task, F1 = 0.80 may still be inadequate; for a very rare or difficult event, a lower F1 may represent valuable improvement.

F1 is commonly used to compare classifiers or decision thresholds when both false positives and false negatives matter. Report precision and recall beside F1 because two models can have similar F1 scores but very different error patterns. For multiclass problems, also state whether the reported score is macro, weighted or per-class F1.

Accuracy can be misleading when one class is much more common. The following example contains about 90% negative cases and 10% positive cases.


In [ ]:
import pandas as pd
from sklearn.datasets import make_classification
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

# Create a reproducible binary dataset with a 90:10 class imbalance.
classification_X, classification_y = make_classification(
    n_samples=1_000,
    n_features=10,
    n_informative=5,
    n_redundant=2,
    weights=[0.90, 0.10],
    class_sep=1.0,
    random_state=42,
)

# Preserve the class proportions in both partitions with stratification.
X_class_train, X_class_test, y_class_train, y_class_test = train_test_split(
    classification_X,
    classification_y,
    test_size=0.25,
    stratify=classification_y,
    random_state=42,
)

# Compare a majority-class baseline with a scaled logistic regression pipeline.
classification_models = {
    "Most-frequent baseline": DummyClassifier(strategy="most_frequent"),
    "Logistic regression": make_pipeline(
        StandardScaler(),
        LogisticRegression(max_iter=1_000, random_state=42),
    ),
}

classification_rows = []
classification_predictions = {}

for model_name, model in classification_models.items():
    model.fit(X_class_train, y_class_train)
    predicted_classes = model.predict(X_class_test)
    classification_predictions[model_name] = predicted_classes
    classification_rows.append(
        {
            "Model": model_name,
            "Accuracy": accuracy_score(y_class_test, predicted_classes),
            "Precision": precision_score(y_class_test, predicted_classes, zero_division=0),
            "Recall": recall_score(y_class_test, predicted_classes, zero_division=0),
            "F1": f1_score(y_class_test, predicted_classes, zero_division=0),
        }
    )

classification_results = pd.DataFrame(classification_rows).set_index("Model")
classification_results.round(3)


The majority-class baseline can achieve high accuracy while finding no positive cases. Precision, recall and F1 reveal this failure.

Metric selection should reflect the consequences of errors:

- prioritise **recall** when missing a positive case is especially costly
- prioritise **precision** when acting on a false positive is especially costly
- use **F1** when false positives and false negatives both matter and a single summary is required
- use the complete confusion matrix when stakeholders need to see the actual error counts

For multiclass classification, calculate per-class metrics and state the averaging method. A **macro average** gives every class equal weight, while a **weighted average** gives larger classes more influence.


In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import ConfusionMatrixDisplay

# Display error counts for the fitted logistic regression model.
ConfusionMatrixDisplay.from_predictions(
    y_class_test,
    classification_predictions["Logistic regression"],
    display_labels=["Negative", "Positive"],
    cmap="Blues",
    colorbar=False,
)
plt.title("Logistic regression confusion matrix")
plt.show()


**Check your understanding:** A screening system must find as many genuine cases as possible, even if some additional cases require manual review. Recall should usually receive more emphasis than precision because false negatives are the more costly error.


## 3. Regression Evaluation Metrics

Regression metrics measure differences between numeric predictions $\hat{y}_i$ and actual values $y_i$. Before calculating them, compare the practical meaning of each metric:

| Metric | Question it answers | Best value | When it is useful |
|---|---|---:|---|
| MAE | On average, how large is the absolute prediction error? | 0.0 | An error in the original target unit is required. |
| MSE | On average, how large is the squared prediction error? | 0.0 | Large errors should receive a strong penalty or the metric is used as an optimisation objective. |
| RMSE | How large is the typical error after giving large mistakes extra influence? | 0.0 | Large errors are especially undesirable, but the result should remain in the target unit. |
| $R^2$ | How much squared error is reduced compared with predicting the mean? | 1.0 | A scale-free summary is needed to compare fit quality on the same target and observations. |

### How Regression Metrics Are Calculated

Let $y_i$ be an actual value, $\hat{y}_i$ its prediction, $\bar{y}$ the mean actual value and $n$ the number of observations.

$$
\text{MAE} = \frac{1}{n}\sum_{i=1}^{n}|y_i-\hat{y}_i|
$$

$$
\text{MSE} = \frac{1}{n}\sum_{i=1}^{n}(y_i-\hat{y}_i)^2
$$

$$
\text{RMSE} = \sqrt{\text{MSE}}
$$

$$
R^2 = 1 - \frac{\sum_{i=1}^{n}(y_i-\hat{y}_i)^2}{\sum_{i=1}^{n}(y_i-\bar{y})^2}
$$

### Interpreting R-squared Values

$R^2$ has an upper limit of **1**, but its lower limit is not fixed; on unseen data its range is $(-\infty, 1]$. A value closer to 1 is generally better:

| $R^2$ value | General meaning |
|---|---|
| 1.00 | perfect predictions |
| 0.70–0.99 | often described as a strong fit, depending on the field and data noise |
| 0.50–0.69 | moderate predictive fit |
| Above 0.00 but below 0.50 | limited improvement over the mean baseline |
| 0.00 | equivalent to predicting the evaluation-set mean under the squared-error definition |
| Below 0.00 | worse than the mean baseline on the evaluated observations |

These bands are an **informal classroom guide, not a universal quality threshold**. An $R^2$ above 0.80 may be considered strong for some applications, while a much lower value may still be useful in noisy human, biological or economic data. Conversely, a very high $R^2$ can result from leakage or from an evaluation set that does not represent future data.

$R^2$ is commonly used to compare regression models evaluated on the same target and the same observations. It should be reported with MAE or RMSE, residual plots and a baseline so the practical error size and error pattern remain visible. Do not compare $R^2$ values from unrelated datasets as though they shared one universal scale.


In [ ]:
import numpy as np
import pandas as pd
from sklearn.datasets import load_diabetes
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

# Load a built-in regression dataset with a continuous disease-progression target.
regression_X, regression_y = load_diabetes(return_X_y=True)

X_reg_train, X_reg_test, y_reg_train, y_reg_test = train_test_split(
    regression_X,
    regression_y,
    test_size=0.25,
    random_state=42,
)

regression_models = {
    "Mean baseline": DummyRegressor(strategy="mean"),
    "Ridge regression": make_pipeline(
        StandardScaler(),
        Ridge(alpha=1.0),
    ),
}

regression_rows = []
regression_predictions = {}

for model_name, model in regression_models.items():
    model.fit(X_reg_train, y_reg_train)
    predicted_values = model.predict(X_reg_test)
    regression_predictions[model_name] = predicted_values
    regression_rows.append(
        {
            "Model": model_name,
            "MAE": mean_absolute_error(y_reg_test, predicted_values),
            "MSE": mean_squared_error(y_reg_test, predicted_values),
            "RMSE": np.sqrt(mean_squared_error(y_reg_test, predicted_values)),
            "R-squared": r2_score(y_reg_test, predicted_values),
        }
    )

regression_results = pd.DataFrame(regression_rows).set_index("Model")
regression_results.round(2)


Lower MAE, MSE and RMSE values are better; higher $R^2$ values are better. A metric comparison is fair only when the models are evaluated on the same observations.

Residuals are defined as $y_i-\hat{y}_i$. A useful model should not show an obvious systematic pattern in its residuals. Curves, funnels or separated groups may indicate bias, changing error variance or missing structure.


In [ ]:
import matplotlib.pyplot as plt

ridge_predictions = regression_predictions["Ridge regression"]
ridge_residuals = y_reg_test - ridge_predictions

figure, axes = plt.subplots(1, 2, figsize=(11, 4))

# Compare predictions with the perfect-prediction diagonal.
axes[0].scatter(y_reg_test, ridge_predictions, alpha=0.7)
plot_min = min(y_reg_test.min(), ridge_predictions.min())
plot_max = max(y_reg_test.max(), ridge_predictions.max())
axes[0].plot([plot_min, plot_max], [plot_min, plot_max], "--", color="black")
axes[0].set_xlabel("Actual target")
axes[0].set_ylabel("Predicted target")
axes[0].set_title("Actual versus predicted")

# Inspect whether residuals remain centred around zero across predictions.
axes[1].scatter(ridge_predictions, ridge_residuals, alpha=0.7)
axes[1].axhline(0, linestyle="--", color="black")
axes[1].set_xlabel("Predicted target")
axes[1].set_ylabel("Residual")
axes[1].set_title("Residual plot")

plt.tight_layout()
plt.show()


**Check your understanding:** Choose MAE when stakeholders need a typical error in the original units. Choose RMSE when large errors deserve extra emphasis. Always interpret the value against a baseline and the practical scale of the target.


## 4. Clustering Evaluation Metrics

Clustering is usually unsupervised, so there may be no correct label for each observation. Evaluation therefore combines numeric evidence with interpretation.

- **Inertia** is the sum of squared distances from observations to their assigned cluster centroids. Lower values indicate more compact clusters, but inertia always decreases as the number of clusters increases.
- **Silhouette score** compares cohesion within a cluster with separation from other clusters. Values range from $-1$ to $1$; larger values generally indicate clearer separation.
- **Stability** asks whether similar groups appear under different random seeds or samples.
- **Usefulness** asks whether the resulting groups are interpretable and support the intended decision or investigation.

External metrics may compare clusters with known reference labels, but those labels must not be used to fit an unsupervised model. A high internal score alone does not prove that the groups are meaningful in practice.


In [ ]:
import pandas as pd
from sklearn.cluster import KMeans
from sklearn.datasets import make_blobs
from sklearn.metrics import silhouette_score
from sklearn.preprocessing import StandardScaler

# Create three visually separable groups for an introductory clustering example.
clustering_X, _ = make_blobs(
    n_samples=450,
    centers=3,
    cluster_std=[1.0, 1.3, 0.8],
    random_state=42,
)

# Standardisation prevents a larger-scale feature from dominating distances.
scaled_clustering_X = StandardScaler().fit_transform(clustering_X)

clustering_rows = []
fitted_cluster_labels = {}

for cluster_count in range(2, 7):
    kmeans_model = KMeans(
        n_clusters=cluster_count,
        n_init=20,
        random_state=42,
    )
    cluster_labels = kmeans_model.fit_predict(scaled_clustering_X)
    fitted_cluster_labels[cluster_count] = cluster_labels
    clustering_rows.append(
        {
            "Clusters": cluster_count,
            "Inertia": kmeans_model.inertia_,
            "Silhouette score": silhouette_score(scaled_clustering_X, cluster_labels),
        }
    )

clustering_results = pd.DataFrame(clustering_rows)
clustering_results.round(3)


In [ ]:
import matplotlib.pyplot as plt

figure, axes = plt.subplots(1, 2, figsize=(11, 4))

# Plot both metrics because each describes a different property of the clusters.
axes[0].plot(clustering_results["Clusters"], clustering_results["Inertia"], marker="o")
axes[0].set_xlabel("Number of clusters, k")
axes[0].set_ylabel("Inertia")
axes[0].set_title("Compactness by cluster count")

axes[1].plot(
    clustering_results["Clusters"],
    clustering_results["Silhouette score"],
    marker="o",
)
axes[1].set_xlabel("Number of clusters, k")
axes[1].set_ylabel("Silhouette score")
axes[1].set_title("Separation by cluster count")

plt.tight_layout()
plt.show()


The best-supported value of $k$ should combine the silhouette score, the change in inertia, visual inspection, stability and practical interpretation. Do not select $k$ by inertia alone because adding clusters mechanically reduces inertia.

**Check your understanding:** If two values of $k$ have similar silhouette scores, investigate whether both solutions remain stable and which grouping is more useful for the stated purpose.


## 5. Underfitting and Overfitting

Model complexity affects how well a model learns useful patterns rather than noise.

| Pattern | Training performance | Validation performance | Interpretation |
|---|---|---|---|
| Underfitting | poor | poor | the model is too simple or lacks useful information |
| Appropriate fit | good | good and reasonably close to training | the learnt pattern generalises |
| Overfitting | very good | noticeably worse | the model has learnt training-specific noise |

A single test score cannot diagnose these patterns. Compare training and validation performance across increasing model complexity. The following demonstration varies the maximum depth of a decision tree.


In [ ]:
import pandas as pd
from sklearn.datasets import make_moons
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier

# Create a non-linear classification problem with moderate noise.
fit_X, fit_y = make_moons(n_samples=800, noise=0.30, random_state=42)

X_fit_train, X_fit_validation, y_fit_train, y_fit_validation = train_test_split(
    fit_X,
    fit_y,
    test_size=0.30,
    stratify=fit_y,
    random_state=42,
)

fit_rows = []

for tree_depth in range(1, 16):
    tree_model = DecisionTreeClassifier(max_depth=tree_depth, random_state=42)
    tree_model.fit(X_fit_train, y_fit_train)
    fit_rows.append(
        {
            "Maximum depth": tree_depth,
            "Training accuracy": accuracy_score(
                y_fit_train,
                tree_model.predict(X_fit_train),
            ),
            "Validation accuracy": accuracy_score(
                y_fit_validation,
                tree_model.predict(X_fit_validation),
            ),
        }
    )

fit_results = pd.DataFrame(fit_rows)
fit_results.round(3)


In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(8, 4.5))
plt.plot(
    fit_results["Maximum depth"],
    fit_results["Training accuracy"],
    marker="o",
    label="Training accuracy",
)
plt.plot(
    fit_results["Maximum depth"],
    fit_results["Validation accuracy"],
    marker="o",
    label="Validation accuracy",
)
plt.xlabel("Maximum tree depth")
plt.ylabel("Accuracy")
plt.title("Training and validation performance by model complexity")
plt.xticks(fit_results["Maximum depth"])
plt.legend()
plt.show()


A shallow tree may underfit because it cannot represent enough of the pattern. As depth increases, training accuracy continues to improve. If validation accuracy stops improving or declines while training accuracy rises, the widening gap is evidence of overfitting.

Possible responses include:

- reduce model complexity or strengthen regularisation
- collect more representative training data
- improve relevant features or remove leakage
- use a simpler model as a baseline
- use cross-validation to check that the pattern is not caused by one fortunate split

Do not choose complexity by repeatedly inspecting the final test set. Use validation evidence during development and keep the test set untouched for final evaluation.


## 6. K-fold Cross-validation

A single validation split can produce a fortunate or unfortunate result. In **k-fold cross-validation**, the training data is divided into $k$ folds:

1. Hold out one fold for validation.
2. Train on the other $k-1$ folds.
3. Evaluate on the held-out fold.
4. Repeat until every fold has been used for validation once.
5. Report the fold scores, their mean and their variation.

For classification, `StratifiedKFold` approximately preserves class proportions in every fold. For regression, ordinary `KFold` is common. Grouped observations require `GroupKFold`, while time-ordered observations require a method such as `TimeSeriesSplit` rather than random folds.

Every operation that learns from data must occur inside each fold. Put scaling, imputation, feature selection and the estimator in one pipeline to prevent leakage.


In [ ]:
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier

# Reuse the original imbalanced classification data, not the earlier test results.
cross_validation = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42,
)

candidate_models = {
    "Logistic regression": make_pipeline(
        StandardScaler(),
        LogisticRegression(max_iter=1_000, random_state=42),
    ),
    "Decision tree": DecisionTreeClassifier(max_depth=5, random_state=42),
}

cv_rows = []

for model_name, model in candidate_models.items():
    # Evaluate the same folds with metrics that reflect the imbalanced task.
    cv_output = cross_validate(
        model,
        classification_X,
        classification_y,
        cv=cross_validation,
        scoring={"accuracy": "accuracy", "recall": "recall", "f1": "f1"},
        return_train_score=True,
    )
    for fold_number in range(cross_validation.get_n_splits()):
        cv_rows.append(
            {
                "Model": model_name,
                "Fold": fold_number + 1,
                "Training F1": cv_output["train_f1"][fold_number],
                "Validation accuracy": cv_output["test_accuracy"][fold_number],
                "Validation recall": cv_output["test_recall"][fold_number],
                "Validation F1": cv_output["test_f1"][fold_number],
            }
        )

cv_fold_results = pd.DataFrame(cv_rows)
cv_fold_results.round(3)


In [ ]:
# Summarise both the typical score and its variation across folds.
cv_summary = (
    cv_fold_results.groupby("Model")
    .agg(
        mean_validation_accuracy=("Validation accuracy", "mean"),
        mean_validation_recall=("Validation recall", "mean"),
        mean_validation_f1=("Validation F1", "mean"),
        std_validation_f1=("Validation F1", "std"),
        mean_training_f1=("Training F1", "mean"),
    )
)

cv_summary.round(3)


Interpret cross-validation results by considering:

- the metric that matches the real error costs
- the mean validation score
- variation across folds
- the gap between training and validation scores
- model simplicity, speed and interpretability

A higher mean with very inconsistent folds may be less dependable than a slightly lower but stable result. Cross-validation supports model selection; it does not replace a final test set that remained independent of all model choices.

**Practice:** Suppose one classifier has mean F1 of 0.78 with a standard deviation of 0.03, while another has mean F1 of 0.80 with a standard deviation of 0.12. Which would you recommend?

**Answer scaffold:** State which model you would choose, compare both the mean and variability, and identify any additional evidence you would examine before making a final recommendation.

**Common pitfall:** Running `StandardScaler().fit_transform(X)` before cross-validation leaks information between folds. Place `StandardScaler()` inside a pipeline so it is fitted separately within each training fold.
